# Project 4: **Build a Deep Research System**
Welcome to project 4! For this project, we shift our focus from tool use and agents to *reasoning* models. You will practice state‑of‑the‑art inference‑time scaling methods such as *Chain‑of‑Thought* prompting and *Tree‑of‑Thoughts*, and briefly explore high-levels of training reasoning models using techniques like **STaR**.


Finally, you will put everything together to build a *deep research agent* that can browse the web, reason over what it finds, and give structured answers.

## Learning Objectives  
* Apply common inference‑time scaling methods: **zero‑shot / few‑shot CoT, self‑consistency, sequential decoding, tree‑of‑thoughts**  
* Gain intuition for **training** reasoning‑capable models following **STaR** approach 
* Build a minimal **deep‑research agent** that combines step‑by‑step reasoning with live web search   
* Practice extending deep-search to a multi-agent system 

## Roadmap  
1. Environment setup  
2. Inference‑time scaling  
   2.1 Few‑shot & zero‑shot CoT  
   2.2 Self‑consistency
   2.3 Sequential revisions  
   2.4 Tree‑of‑Thought
3. STaR for training models for reasoning  
4. Deep-research agent  
5. (Optional) Multi-agent deep-research

# 1‑ Environment setup

## 1.1- Conda environment

Before we start coding, you need a reproducible setup. Open a terminal in the same directory as this notebook and run:

```bash
# Create and activate the conda environment
conda env create -f environment.yaml && conda activate deep_research

# Register this environment as a Jupyter kernel
python -m ipykernel install --user --name=deep_research --display-name "deep_research"
```
Once this is done, you can select "deep_research" from the Kernel → Change Kernel menu in Jupyter or VS Code.

## 1.2 Ollama setup

In this project we use the `llama3.2:3b` and `deepseek-r1:8b` models. You can try other smaller or larger reasoning LLMs such as `qwen2.5:3b-instruct` or `phi4-mini` to compare performance. Explore available models here: https://ollama.com/library.

```bash
ollama pull llama3.2:3b
ollama pull deepseek-r1:8b
# Additional small reasoning models to compare
# ollama pull qwen2.5:3b-instruct
# ollama pull phi4-mini

```

`ollama pull` downloads the model so you can run it locally without API calls.

---  
# 2‑ Inference‑time scaling

Inference-time scaling refers to techniques that make an existing model reason better without retraining it. Instead of changing the model’s weights, we achieve reasoning capability by adjusting how we prompt, sample, or aggregate LLM's outputs.

In this section, we’ll explore several inference-time strategies that improve reasoning quality using a non-reasoning base model. You will experiment with and compare methods such as:

- Few-shot Chain-of-Thought (CoT)
- Zero-shot CoT
- Self-consistency
- Sequential revision
- Tree-of-Thoughts (ToT)

### 2.1: Few‑Shot CoT
Few-shot prompting helps a model reason by showing one or multiple examples before asking a new question. By observing the pattern of reasoning and final answers, the model learns how to structure its own reasoning process on the new input.

In this exercise, you will create a prompt that includes a few example Q&A pairs demonstrating step-by-step reasoning. Then, you will feed a new question and see the model’s output.

In [1]:
# Step 1: Write a few examples showing reasoning steps
# Step 2: Write your new question
# Step 3: Concatenate examples + new question into a single prompt
# Step 4: Call your Ollama or OpenAI client to get a response from llama3.2:3b # e.g., client.chat.completions.create(...)
# Step 5: Print the final answer

from openai import OpenAI

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")


USER_QUESTION = """# Example
Q: There are 2 boxes with 3 balls each. How many balls are there?
A: 2x3 = 6. So, 6.

# Now solve this
Q: There are 3 red bags with 5 apples each and 2 blue bags with 7 apples each. How many apples are there in total?
A:"""

response = client.chat.completions.create(
    model="gemma3:1b", 
    messages=[
        {"role": "user", "content": USER_QUESTION}
    ]
)

print(response.choices[0].message.content)

3 red bags * 5 apples/bag = 15 apples.
2 blue bags * 7 apples/bag = 14 apples.

Total apples = 15 + 14 = 29 apples.

So, the answer is 29.


### (Optional) Few-shot CoT on GPT2
GPT-2 is a pre-trained language model without instruction tuning. It continues text rather than answering questions. In this section, you'll try the exact same CoT pattern on GPT-2 and observe what happens. The goal is to test whether few-shot CoT alone can elicit structured reasoning from a non-chat LLM.

In [3]:
import os
import torch
from transformers import pipeline

# Step 1: Load GPT-2 text-generation from huggingface (https://huggingface.co/docs/transformers/en/model_doc/gpt2)
# Step 2: Write 1–2 few-shot reasoning examples (short, explicit steps + final answer in your own unique format)
# Step 3: Append a new test question after the examples to form one prompt string
# Step 4: Generate 1–3 completions with different decoding settings (e.g., greedy vs. top-k)
# Step 5: Print raw outputs; check if steps are followed and if the final answer is correct

pipeline = pipeline(task="text-generation", model="openai-community/gpt2", dtype=torch.float16, device=0)

USER_QUESTION = """# Example
Q: There are 2 boxes with 3 balls each. How many balls are there?
A: 2x3 = 6. So, 6.

# Now solve this
Q: There are 3 red bags with 5 apples each and 2 blue bags with 7 apples each. How many apples are there in total?
A:"""

pipeline(USER_QUESTION)

Device set to use mps:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': '# Example\nQ: There are 2 boxes with 3 balls each. How many balls are there?\nA: 2x3 = 6. So, 6.\n\n# Now solve this\nQ: There are 3 red bags with 5 apples each and 2 blue bags with 7 apples each. How many apples are there in total?\nA: 1x1 = 5. So, 5 apples.\n\n# Now solve this example\n\nQ: There are 3 boxes with 3 apples each and 2 blue bags with 7 apples each. How many apples are there in total?\n\nA: 1x1 = 5. So, 5 apples.\n\n# Now solve this example\n\nQ: There are 3 boxes with 3 apples each and 2 blue bags with 7 apples each. How many apples are there in total?\n\nA: 1x1 = 5. So, 5 apples.\n\n# Now solve this example\n\nQ: There are 3 boxes with 3 apples each and 2 blue bags with 7 apples each. How many apples are there in total?\n\nA: 1x1 = 5. So, 5 apples.\n\n# Now solve this example\n\nQ: There are 3 boxes with 3 apples each and 2 blue bags with 7 apples each. How many apples are there in total?\n\nA: 1x1 = 5. So, 5 apples.\n\n# Now solve this example\n\n

### 2.2: Zero‑Shot Chain‑of‑Thought
Zero-shot CoT encourages the model to reason without examples by adding a short cue such as “Let’s think step by step.” This simple phrase often activates the model’s latent reasoning ability even when no demonstrations are provided. It serves as a baseline to compare with few-shot and other inference-time scaling methods.

In [4]:
from openai import OpenAI

# Step 1: Write the question and a zero-shot CoT cue (e.g., "Let's think step by step.")
# Step 2: Build a single prompt string that includes brief role guidance plus the question
# Step 3: Call your Ollama or OpenAI client to get a response from llama3.2:3b  # e.g., client.chat.completions.create(...)
# Step 4: Print the chain and the final answer

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")


USER_QUESTION = """
There are 3 red bags with 5 apples each and 2 blue bags with 7 apples each. 
How many apples are there in total? 
Let's think step by step.
"""

response = client.chat.completions.create(
    model="gemma3:1b", 
    messages=[
        {"role": "user", "content": USER_QUESTION}
    ]
)

print(response.choices[0].message.content)

Okay, let's solve this step by step:

*   **Red bags:** 3 bags * 5 apples/bag = 15 apples
*   **Blue bags:** 2 bags * 7 apples/bag = 14 apples
*   **Total apples:** 15 apples + 14 apples = 29 apples

**Answer:** There are a total of 29 apples.


### 2.3 Self‑Consistency
Self-consistency enhances reasoning accuracy by sampling multiple independent reasoning paths for the same question instead of relying on a single deterministic answer. Each run may follow a slightly different logical chain, and the diversity helps correct individual mistakes. After generating several reasoning traces, you then aggregate the final answers using majority voting.

This approach is especially useful when tasks involve multi-step reasoning or arithmetic, where single-path outputs may be incorrect.

In [ ]:
from openai import OpenAI
import re, collections

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")
MODEL = "llama3.2:3b"

SYSTEM_PROMPT = """
    You are a helpful AI assistant that excels at problem-solving.
    Think step by step to come up with an answer.
    When you get the answer, give the response as the just final answer after the keyword 'ANSWER:' (no full stops or anything else).
    """

def cot_answer(question, temperature=1.0):
    # Generate a step-by-step reasoning chain for the given question and extract the final answer.
    response = client.chat.completions.create(
        model=MODEL, 
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]
    )
    
    return response.choices[0].message.content


def self_consistent(question, n=10):
    # Run multiple reasoning chains and select the most frequent final answer by majority voting.
    responses = [cot_answer(question).split("ANSWER:")[1] for _ in range(n)]
    
    print(responses)
    counts = collections.Counter(responses)
    max_key = max(counts, key=counts.get)
    
    return max_key, counts[max_key]


question = "What is the square root of 144?"
winner, counter = self_consistent(question, 4)
print("Votes:", counter)
print("Chosen answer:", winner)

[' 12', ' 12', ' 12', ' 12']
Votes: 4
Chosen answer:  12


### 2.4: Sequential Revision

Sequential revision iteratively improves an answer by generating a first draft, critiquing it, and producing revised drafts that condition on prior answers. Each round should be short and focused, so improvements accumulate without drifting from the question.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")
MODEL = "llama3.2:3b"


INITIAL_SYSTEM_PROMPT = """
    You are a helpful AI assistant. 
    You are to provide the first draft for the response that can be improved upon.
    """
    
REVISION_SYSTEM_PROMPT = """
    You are to take the previous revision, and improve the draft provided and respond with an improved draft.
    """

def sequential_revision(question: str, max_steps: int = 3) -> str:
    # Generate an initial draft answer, then iteratively refine it by conditioning each revision on the previous one.
    # Step 1: Ask the model to produce the first draft for the given question
    # Step 2: Loop for max_steps-1 times, each time feeding the last draft back to the model with a request to revise
    # Step 3: Print each draft to observe how the answer evolves
    # Step 4: Return the final improved draft
    first_response_obj = client.chat.completions.create(
        model=MODEL, 
        messages=[
            {"role": "system", "content": INITIAL_SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]
    )
    
    response = first_response_obj.choices[0].message.content
    print(f"response 1: {response}")
    
    for i in range(max_steps - 1):
        next_response_obj = client.chat.completions.create(
            model=MODEL, 
            messages=[
                {"role": "system", "content": REVISION_SYSTEM_PROMPT},
                {"role": "user", "content": question},
                {"role": "user", "content": response}
            ]
        )
        
        response = next_response_obj.choices[0].message.content
        print(f"response {i + 2}: {response}")
        
    return response


# Step 1: Define a question that benefits from multi-step reasoning
# Step 2: Call sequential_revision(question, max_steps)
# Step 3: Print the final output

question = "Give me a 50 word snippet that I can send to my girl-friend to make her feel special."
revised_response = sequential_revision(question, 5)
print("Revised Response:", revised_response)

response 1: Here's a 50-word snippet:

"Hey love, just wanted to let you know how amazing you are. From the moment I met you, I knew you were someone special. Your kind heart and beautiful smile light up my world. I'm so grateful to have you by my side. You make every day brighter."
response 2: Here's a revised 50-word snippet that conveys a similar message with a slightly different tone:

"You're the sunshine that brightens every day, my love. Since the moment we met, your kindness and beauty captivated me. I'm so grateful to have you as my partner in life – my rock, my confidante, and my forever home. You make my world a better place."

This revised snippet aims to make her feel special by using more poetic language and emphasizing the idea of her being his "forever home".
response 3: Here's an improved draft of the 50-word snippet:

"Infinite moments with you are etched in my memory, treasured as precious gems. Your kindness, beauty, and laughter are my guiding lights. You're the mi

### 2.5 Tree‑of‑Thoughts
Tree-of-Thoughts reframes reasoning as a search process rather than a single forward chain.
Instead of producing one linear sequence of thoughts, the model generates multiple candidate thoughts at each step, evaluates their promise, and then expands only the best few. This allows exploration of different reasoning paths before committing to a final answer, similar to how humans brainstorm, prune, and refine ideas.


In this section, you’ll experiment with two simplified versions of ToT:
1. Word Ladder puzzle solver: a small example where each “thought” is a candidate word transition.
2. Generic ToT search (depth 2, width 2): a minimal logic to expand, evaluate, and select reasoning branches

In [2]:
###### Word Ladder Puzzle ##########

def neighbors(word, vocabulary):
    # Generate all valid one-letter mutations of 'word' that exist in 'vocabulary' and return them.
    for i, c1 in enumerate(word):
        for c2 in 'abcdefghijklmnopqrstuvwxyz':
            if c1 != c2:
                candidate = word[:i] + c2 + word[i + 1:]
                if candidate in vocabulary:
                    yield candidate


def tree_of_thought(start, goal, vocab, max_depth=5, beam_width=4):
    # Search over partial thoughts (paths) using a small beam.
    # Step 1: Initialize the frontier with a single path [start]
    # Step 2: For each depth, expand each path by one neighbor from 'neighbors'
    # Step 3: Score paths by edit distance between last word and 'goal' (smaller is better)
    # Step 4: Keep the top 'beam_width' paths and stop early if any reaches 'goal'
    # Step 5: Return the best goal-reaching path or None
    
    frontier = [[start]]
    
    for depth in range(max_depth):
        candidates = []
        
        for path in frontier:
            for nxt in neighbors(path[-1], vocab):
                if nxt in path:
                    continue
            
                candidates.append(path + [nxt])

        scored = sorted(candidates, key=lambda p: sum(a != b for a, b in zip(p[-1], goal)))
        frontier = scored[:beam_width]
        
        if any(p[-1] == goal for p in frontier):
            return [p for p in frontier if p[-1] == goal][0]
    
    return None


vocab = {"hit","dot","cog","log","dog","lot","lit","hot"}
print(tree_of_thought("hit", "cog", vocab)) # one candidate solution: ['hit', 'hot', 'dot', 'dog', 'cog']


['hit', 'hot', 'dot', 'dog', 'cog']


In [10]:
###### Generic ToT Search ##########

import re
from openai import OpenAI

client = OpenAI(api_key = "ollama", base_url = "http://localhost:11434/v1")
MODEL = "llama3.2:3b"

def propose_thoughts(question, state, k=2):
    # Propose up to k next “thoughts” that extend the current partial solution/state.
    # Steps: build a short prompt with problem + current state; call your client with n=k. Then return a list of stripped strings (≤ k).
    prompt = f"""You are exploring solutions.
                Question: {question}
                Current State: {state}
                
                Propose atmost {k} different next thoughts."""
    response_obj = client.chat.completions.create(
        model=MODEL, 
        messages=[{"role": "user", "content": prompt}],
        n=k
    )
    
    return [c.message.content.strip() for c in response_obj.choices]


def score_state(question, state):
    # Score how promising a partial solution is on a 1–10 scale (higher is better).
    # Steps: build a rating prompt; call the model; parse the first integer 1–10;
    prompt = f"""
        You are scoring the partial solutions on a scale of 1 to 10.
        Question: {question}
        State: {state}
    """
    
    response_obj = client.chat.completions.create(
        model=MODEL, 
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    try:
        return int(re.findall(r"\d+", response_obj.choices[0].message.content[0]))
    except Exception:
        return 5 # neutral feedback


def tree_of_thoughts(question, depth=2, width=2):
    # Run a tiny ToT search: expand states with propose_thoughts, score with score_state, keep top-k at each depth.
    # Steps: initialize frontier=[("", 0)]; for each depth, expand each state with k=width thoughts; score each; sort by score desc; keep top 'width'; return best state and score.
    frontier = [("", 0)]
    
    for _ in range(depth):
        new_frontier = []
        
        for state, _ in frontier:
            for thought in propose_thoughts(question, state, k=width):
                new_state = (state + "\n" + thought).strip()
                score = score_state(question, new_state)
                new_frontier.append((new_state, score))
        
        # keep top k
        new_frontier.sort(key=lambda x: x[1], reverse=True)
        frontier = new_frontier[:width]
    
    best_state, best_score = frontier[0]
    return best_state, best_score


question = "Design a plan for a weekend science workshop for 12-year-olds."
solution, score = tree_of_thoughts(question)

print(f"Best solution (score {score}):\n{solution}")

Best solution (score 5):
Here are two possible directions for designing a plan for a weekend science workshop for 12-year-olds:

**Direction 1: Hands-on Experimentation and Teamwork**

* Theme: "The Science of Innovation"
* Objective: To encourage curiosity, critical thinking, and collaboration among participants while exploring the principles of science, technology, engineering, and mathematics (STEM).
* Activities:
 + Designing and building a simple bridge using everyday materials
 + Conducting experiments with different types of magnets and magnetic properties
 + Creating a model of a sustainable community using recycled materials
 + Collaborative puzzle-solving to understand basic physics concepts
* Facilitators: Science instructors or engineers who can guide participants through the activities, provide guidance, and facilitate discussions.

**Direction 2: Hands-on Science Exploration and Discovery**

* Theme: "The Wonder of Nature"
* Objective: To spark curiosity, creativity, and 

---  
# 3‑ Training Models for Reasoning

### 3.1: CoT Training
Chain-of-Thought (CoT) training conditions the model on explicit rationales during fine-tuning. Instead of teaching the model to output only the final answer, we train on (question, rationale, answer) so the model learns to internalize multi-step reasoning patterns. A practical recipe is STaR (Self-Taught Reasoner), which uses a stronger teacher model to bootstrap rationales that a smaller student can learn from.

For tasks that require multi-hop reasoning, models fine-tuned on rationales often achieve higher accuracy and are more stable at inference time than models trained on direct answers only. 

Training a full language model is beyond the scope of this notebook, but here is the high-level workflow followed by a short pseudocode:
- Collect questions: Prepare a dataset of questions and correct answers.
- Generate rationales: Use a strong LLM to produce step-by-step reasoning ending with the correct answer.
- Filter and clean: Discard incorrect or low-quality rationales.
- Prepare training data: Format triples (question, rationale, answer) for supervised fine-tuning.
- Fine-tune: Fine-tune the LLM on rationales.
- Iterate: Refine prompts, improve data quality, and retrain for stronger reasoning.

In [ ]:
# Pseudocode (STaR loop)
# for round in 1 ... iters:
    # STEP 1: self-generate reasoning (teacher creates rationale + answer)
    # STEP 2: keep only correct, high-quality traces
    # STEP 3: fine-tune student on (question, rationale, answer) data

### 3.2: ORM vs PRM + RL
Training a Reward Model (RM) allows large language models to be improved through reinforcement learning (RL). Instead of fine-tuning directly on examples, we train a separate model that can score or rank model outputs, and use those scores as feedback signals to refine the policy model.

Two main reward modeling approaches are ORM (predicts a scalar reward for the final answer) and PRM (evaluates the reasoning steps instead of just the outcome)



| Approach | Typical loss | When to use |
|-----------|-------------|-------------|
|*Outcome Reward Model* | Predict scalar reward | Easy to collect training data using verifiers |
|*Process Reward Model* | Predict rewards per step | Difficult to collect training data but more accurate |
| *RLHF* | Use RM as reward in **RL** fine‑tuning | Aligns policy with human signals | Aligns model policy with human or synthetic preferences




In [ ]:
# for round = 1 ... iters:
    # STEP 1:  Generate reasoning
        # sample a minibatch of questions
        # policy roll‑out (actions + log‑probs)
    # STEP 2:  Score the trajectory
        # ORM: scalar reward for the final answer / PRM: scalar reward for the thought process
    # STEP 3:  Reinforce the policy (PPO)

---  
# 4‑ A Deep Research Agent

A deep-research agent pairs a reasoning model (e.g., deepseek-r1) with external tools for web search and retrieval. We will follow the ReAct pattern: the model writes short thoughts, decides when to call tools, reads observations, and continues reasoning until it can answer or reaches a step limit.

We now combine a **search tool** with a reasoning model (e.g., `deepseek-r1`) in a multi-step setup. We follow the *ReAct* pattern (reason → tool → observation):

1. The model reasoins and decides to use tools
2. The agent searches and feed condensed snippets back as context
3. Iterate until the model answers or hits a step limit

We use `AgentType.OPENAI_FUNCTIONS`, which hides the loop inside the LangChain agent.

In [11]:
from ddgs import DDGS
from langchain.tools import Tool

def ddg_search(query: str, k: int = 5) -> str:
    # Use DDGS to run a simple web search and return joined snippets.
    with DDGS() as ddgs:
        results = [hit["body"] for hit in str(ddgs.text(query, max_results=k))]
    return "\n".join(results)

search_tool = Tool(
    name="DuckDuckGo Search",
    func=ddg_search,
    description="Search the public web. Input: a plain English query. Returns: concatenated snippets."
)


In [12]:
from langchain.agents import initialize_agent, AgentType
from langchain_community.chat_models import ChatOllama

MODEL = "deepseek-r1:8b"
question = "What are the best resources to learn machine learning in 2025?"

# Step 1: Initialize the reasoning model via ChatOllama
llm = ChatOllama(model=MODEL, temperature=0.2)

# Step 2: Build the agent with tool access (DuckDuckGo Search) and function-calling interface (initialize_agent)
agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)


# Step 3: Ask a query and let the agent search + reason to produce an answer
result = agent.invoke({"input": question})
print(result["output"])

/var/folders/0t/nnx__l0j50g0ln6_bqdz3m340000gn/T/ipykernel_1928/2743880544.py:8: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=MODEL, temperature=0.2)
/var/folders/0t/nnx__l0j50g0ln6_bqdz3m340000gn/T/ipykernel_1928/2743880544.py:11: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Mi



> Entering new AgentExecutor chain...

Okay, finding the "best" resources for learning Machine Learning (ML) in 2025 requires looking at what's likely to be available and effective based on current trends and foundational knowledge. Since 2025 is in the future, we can't know for certain, but we can project based on today's landscape and anticipated developments.

Here's a breakdown of likely effective resources, categorized by format and focus:

## I. Online Courses & Platforms (Highly Evolving but Foundational Concepts Remain)

1.  **Introductory & Foundational Courses (Likely to remain core):**
    *   **Coursera (Stanford, Andrew Ng):** "Machine Learning" Specialization (by Andrew Ng) is likely to be updated but remain a cornerstone. Expect newer courses covering specific areas like Deep Learning (DeepLearning.AI Specialization), LLMs, Causal Inference.
    *   **edX (MIT, Harvard, Berkeley):** Excellent courses from top universities, often covering similar ground to Coursera but 

# Optional (Multi-agent Deep Research)
Instead of a single multi-step agent, you can design multiple collaborating agents such as a Planner, Searcher, Summarizer, and Verifier that pass information and refine each other’s outputs. This setup improves robustness, diversity of reasoning, and division of labor.

Try building a simple setup with 2–3 agents that share goals and messages, for example Planner → Researcher → Writer.

In [ ]:
def parallel_research(query, n=3):
    # Run n independent research runs in parallel and return their answers.
    # Steps: use ThreadPoolExecutor; submit n calls to your agent/search pipeline; gather results in order.
    """
    YOUR CODE HERE
    """

answers = parallel_research("What are the best resources to learn ML in 2025?")
for i,a in enumerate(answers,1):
    print(f"[Run {i}] {a[:200]}…")

## 🎉 Congratulations!

* Practised various inference‑time reasoning methods
* Gained intuition about training reasoning models
* You have built a **deep-research agent**: reasoning model like deep-seek r1 + ReAct-style agent + tool use (web search)
* Try adding more tools, and extending the deep-research to a multi-agent system: many agents researching web in parallel.


👏 **Great job!** Take a moment to celebrate. The techniques you implemented here power many production agents and chatbots.